In [1]:
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent
CORPUS_PATH = PROJECT_ROOT / "data" / "corpus" / "hot_beverages.json"

print(f"Project root: {PROJECT_ROOT}")
print(f"Corpus path: {CORPUS_PATH}")

Project root: /voc/work/W6_Gorthi
Corpus path: /voc/work/W6_Gorthi/data/corpus/hot_beverages.json


Query
  ↓
Embed query
  ↓
Compare against all 20 chunks
  ↓
Calculate similarity
  ↓
Remove scores < threshold
  ↓
Sort remaining chunks
  ↓
Take Top-K

In [2]:
with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    documents = json.load(f)

print(f"Loaded {len(documents)} documents.")

Loaded 10 documents.


In [3]:
print("First document:")
print(documents[0])

print("\nDocument ID:")
print(documents[0]["id"])

print("\nDocument text:")
print(documents[0]["text"])

First document:
{'id': 'coffee_espresso', 'text': 'Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.'}

Document ID:
coffee_espresso

Document text:
Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.


In [4]:
def chunk_text(text, size=200, overlap=40):
    """Split text into overlapping character-based chunks."""

    if overlap >= size:
        raise ValueError("overlap must be smaller than size")

    if len(text) <= size:
        return [text]

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + size, len(text))
        chunks.append(text[start:end])

        if end == len(text):
            break

        start = end - overlap

    return chunks

In [5]:
sample_text = documents[0]["text"]

chunks = chunk_text(sample_text)

print(f"Original document length: {len(sample_text)} characters")
print(f"Number of chunks: {len(chunks)}")

for i, chunk in enumerate(chunks):
    print(f"\nChunk {i}: {len(chunk)} characters")
    print(chunk)

Original document length: 298 characters
Number of chunks: 2

Chunk 0: 200 characters
Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3

Chunk 1: 138 characters
y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.


In [6]:
overlap_text = chunks[0][-40:]

print("Last 40 characters of Chunk 0:")
print(repr(overlap_text))

print("\nFirst 40 characters of Chunk 1:")
print(repr(chunks[1][:40]))

print("\nOverlap is identical:")
print(chunks[0][-40:] == chunks[1][:40])

Last 40 characters of Chunk 0:
'y 25 to 30 millilitres and takes 25 to 3'

First 40 characters of Chunk 1:
'y 25 to 30 millilitres and takes 25 to 3'

Overlap is identical:
True


In [7]:
all_chunks = []

for document in documents:
    document_chunks = chunk_text(document["text"])

    for chunk_index, chunk in enumerate(document_chunks):
        all_chunks.append({
            "chunk_id": f"{document['id']}#{chunk_index}",
            "source_id": document["id"],
            "text": chunk
        })

print(f"Documents: {len(documents)}")
print(f"Total chunks: {len(all_chunks)}")

Documents: 10
Total chunks: 20


Step 8 — Inspect the chunks

Before moving to embeddings, let's inspect what we've actually created.

We want to answer two questions:

What does an individual chunk look like?
Can we trace a chunk back to its original document?

In [8]:
for chunk in all_chunks[:5]:
    print(f"Chunk ID : {chunk['chunk_id']}")
    print(f"Source   : {chunk['source_id']}")
    print(f"Length   : {len(chunk['text'])} characters")
    print(f"Text     : {chunk['text']}")
    print("-" * 80)

Chunk ID : coffee_espresso#0
Source   : coffee_espresso
Length   : 200 characters
Text     : Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3
--------------------------------------------------------------------------------
Chunk ID : coffee_espresso#1
Source   : coffee_espresso
Length   : 138 characters
Text     : y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.
--------------------------------------------------------------------------------
Chunk ID : coffee_beans#0
Source   : coffee_beans
Length   : 200 characters
Text     : Coffee beans come primarily from two species: Arabica and Robusta. Arabica accounts for about 60 percent of world production and is prized for its smoother, more nuanced flavour. Robusta contains roug
------------------------

Step 9 — Count chunks per source

After inspecting a few chunks, let's verify that every document generated exactly two chunks.

In [9]:
from collections import Counter

chunk_counts = Counter(chunk["source_id"] for chunk in all_chunks)

print("Chunks per source document:")
for source_id, count in chunk_counts.items():
    print(f"  {source_id:25s} {count} chunks")

Chunks per source document:
  coffee_espresso           2 chunks
  coffee_beans              2 chunks
  coffee_brewing            2 chunks
  tea_green                 2 chunks
  tea_black                 2 chunks
  tea_oolong                2 chunks
  chocolate_traditional     2 chunks
  chocolate_powder          2 chunks
  chocolate_history         2 chunks
  milk_latte                2 chunks


Cell 10 — Embedding setup

Add a new Code cell after your current chunk-count cell.

In [10]:
import os
import numpy as np
from openai import OpenAI

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"

client = OpenAI()

EMBED_MODEL = "text-embedding-3-small"
CHAT_MODEL = "gpt-4o-mini"

print("Embedding setup is ready.")
print(f"Embedding model: {EMBED_MODEL}")

Embedding setup is ready.
Embedding model: text-embedding-3-small


Cell 11 — Create embed_batch()

In [11]:
def embed_batch(texts):
    """Create embeddings for a list of texts in one API call."""
    
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    
    return [item.embedding for item in response.data]


# Extract the text from all 20 chunks
chunk_texts = [chunk["text"] for chunk in all_chunks]

# Create embeddings for all 20 chunks in ONE API call
vectors = embed_batch(chunk_texts)

# Attach each embedding to its corresponding chunk
for chunk, vector in zip(all_chunks, vectors):
    chunk["vector"] = vector

print(f"Embedded {len(vectors)} chunks.")
print(f"Each embedding has {len(vectors[0])} dimensions.")
print(f"First 8 dimensions of chunk 0:")
print(vectors[0][:8])

Embedded 20 chunks.
Each embedding has 1536 dimensions.
First 8 dimensions of chunk 0:
[-0.0325927734375, -0.03851318359375, -0.0031604766845703125, -0.06585693359375, 0.009307861328125, -0.038299560546875, -0.029296875, 0.053619384765625]


small verification cell:

In [12]:
chunk = all_chunks[0]

print("Chunk ID:", chunk["chunk_id"])
print("Source ID:", chunk["source_id"])
print("Text preview:", chunk["text"][:100])
print("Vector dimensions:", len(chunk["vector"]))
print("First 5 vector values:", chunk["vector"][:5])

Chunk ID: coffee_espresso#0
Source ID: coffee_espresso
Text preview: Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure t
Vector dimensions: 1536
First 5 vector values: [-0.0325927734375, -0.03851318359375, -0.0031604766845703125, -0.06585693359375, 0.009307861328125]


Cell 12 — Define cosine similarity

In [13]:
import numpy as np

def cosine(a, b):
    va = np.array(a)
    vb = np.array(b)

    return float(
        np.dot(va, vb)
        / (np.linalg.norm(va) * np.linalg.norm(vb))
    )

separate test cell:

In [14]:
print("Same direction:", cosine([1, 0], [1, 0]))
print("Opposite direction:", cosine([1, 0], [-1, 0]))
print("Perpendicular:", cosine([1, 0], [0, 1]))

Same direction: 1.0
Opposite direction: -1.0
Perpendicular: 0.0


Next Cell — Embed a question

In [15]:
QUERY = "how is espresso made?"

query_vector = embed_batch([QUERY])[0]

print("Query:", QUERY)
print("Query vector dimensions:", len(query_vector))
print("First 5 values:", query_vector[:5])

Query: how is espresso made?
Query vector dimensions: 1536
First 5 values: [-0.053741455078125, -0.06243896484375, 0.0282135009765625, -0.06591796875, 0.0216827392578125]


Compare the query with all 20 chunks

In [16]:
scored = []

for chunk in all_chunks:
    score = cosine(query_vector, chunk["vector"])
    scored.append((score, chunk))

scored.sort(key=lambda pair: pair[0], reverse=True)

scored = []

for chunk in all_chunks:
    score = cosine(query_vector, chunk["vector"])
    scored.append((score, chunk))

scored.sort(key=lambda pair: pair[0], reverse=True)

print(f"Query: {QUERY}\n")

print(f"{'Rank':<6}{'Score':<10}{'Chunk ID':<30}Preview")
print("-" * 90)

for rank, (score, chunk) in enumerate(scored, start=1):
    preview = chunk["text"][:70].replace("\n", " ")
    print(f"{rank:<6}{score:<10.3f}{chunk['chunk_id']:<30}{preview}...")

Query: how is espresso made?

Rank  Score     Chunk ID                      Preview
------------------------------------------------------------------------------------------
1     0.669     coffee_espresso#0             Espresso is a concentrated form of coffee made by forcing hot water un...
2     0.575     coffee_espresso#1             y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso...
3     0.570     milk_latte#1                  espresso to five parts milk. A cappuccino uses the same espresso base ...
4     0.565     milk_latte#0                  A caffè latte is made with one shot of espresso and around 200 millili...
5     0.540     coffee_beans#1                e nuanced flavour. Robusta contains roughly twice as much caffeine and...
6     0.464     coffee_brewing#1              light-bodied cup. French press coffee, in contrast, steeps coarse grou...
7     0.429     coffee_brewing#0              Pour-over coffee uses a filter cone to drip near-boiling wa

Top-K

In [17]:
def retrieve(query, chunks, k=3):
    query_vector = embed_batch([query])[0]

    scored = []

    for chunk in chunks:
        score = cosine(query_vector, chunk["vector"])
        scored.append((score, chunk))

    scored.sort(key=lambda pair: pair[0], reverse=True)

    return [
        {**chunk, "score": score}
        for score, chunk in scored[:k]
    ]

Test it

In [18]:
top3 = retrieve("how is espresso made?", all_chunks, k=3)

print("Top 3 results:\n")

for rank, hit in enumerate(top3, start=1):
    print(f"{rank}. {hit['chunk_id']}  score={hit['score']:.3f}")
    print(f"   {hit['text']}")
    print()

Top 3 results:

1. coffee_espresso#0  score=0.669
   Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3

2. coffee_espresso#1  score=0.575
   y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.

3. milk_latte#1  score=0.570
   espresso to five parts milk. A cappuccino uses the same espresso base but has equal parts milk and foam, giving it a lighter, airier texture.



Cell 13 — Build the grounded prompt

In [19]:
SYSTEM = (
    "You are a helpful assistant. Answer the user's question using ONLY the "
    "provided context. If the context does not contain the answer, say so "
    "plainly. Cite the source id in square brackets after any fact you use."
)


def build_prompt(question, retrieved):
    context = "\n\n".join(
        f"[{hit['chunk_id']}]\n{hit['text']}"
        for hit in retrieved
    )

    user_message = (
        f"Context:\n{context}\n\n"
        f"---\n\n"
        f"Question: {question}"
    )

    return SYSTEM, user_message

Now test the prompt

In [20]:
system_message, user_message = build_prompt(
    "how is espresso made?",
    top3
)

print("========== SYSTEM ==========")
print(system_message)

print("\n========== USER ==========")
print(user_message)

========== SYSTEM ==========
You are a helpful assistant. Answer the user's question using ONLY the provided context. If the context does not contain the answer, say so plainly. Cite the source id in square brackets after any fact you use.

========== USER ==========
Context:
[coffee_espresso#0]
Espresso is a concentrated form of coffee made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 3

[coffee_espresso#1]
y 25 to 30 millilitres and takes 25 to 30 seconds to extract. Espresso forms the base of drinks like the latte, cappuccino, and americano.

[milk_latte#1]
espresso to five parts milk. A cappuccino uses the same espresso base but has equal parts milk and foam, giving it a lighter, airier texture.

---

Question: how is espresso made?


Now we're ready for Generation
The next cell will finally call the LLM.

In [21]:
response = client.chat.completions.create(
    model=CHAT_MODEL,
    temperature=0.0,
    messages=[
        {
            "role": "system",
            "content": system_message
        },
        {
            "role": "user",
            "content": user_message
        }
    ]
)

answer = response.choices[0].message.content

print("Answer:")
print(answer)

Answer:
Espresso is made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract [coffee_espresso#0].


Next cell — ask_rag()

In [22]:
def ask_rag(question, chunks, k=3):
    # Step 1: retrieve the most relevant chunks
    retrieved = retrieve(question, chunks, k=k)

    # Step 2: build the grounded prompt
    system_message, user_message = build_prompt(
        question,
        retrieved
    )

    # Step 3: ask the LLM
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    # Step 4: extract the generated answer
    answer = response.choices[0].message.content

    # Step 5: return answer and source IDs
    return {
        "question": question,
        "answer": answer,
        "sources": [hit["chunk_id"] for hit in retrieved]
    }

ask_rag(question)
       │
       ├── retrieve()
       │       ↓
       │    Top-K chunks
       │
       ├── build_prompt()
       │       ↓
       │    Grounded prompt
       │
       ├── LLM
       │       ↓
       │    Answer
       │
       └── sources

Test it

Then create one more cell:

In [23]:
result = ask_rag(
    "how is espresso made?",
    all_chunks,
    k=3
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Question:
how is espresso made?

Answer:
Espresso is made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract [coffee_espresso#0].

Sources:
['coffee_espresso#0', 'coffee_espresso#1', 'milk_latte#1']


1. Corpus
10 documents
2. Chunking
10 documents
      ↓
20 chunks

with:

chunk size = 200
overlap = 40
3. Embedding
20 chunks
      ↓
text-embedding-3-small
      ↓
20 × 1536-dimensional vectors
4. Query embedding
"how is espresso made?"
      ↓
1536-dimensional vector
5. Similarity
Query vector
      ↓
cosine similarity
      ↓
20 scores
6. Retrieval
20 scores
      ↓
sort descending
      ↓
Top-K = 3
7. Prompt construction
System instruction
+
Retrieved context
+
Question
8. Generation
Prompt
   ↓
gpt-4o-mini
   ↓
Grounded answer
9. Sources
Retrieved chunks
      ↓
chunk IDs

In [24]:
result = ask_rag(
    "Who invented the telephone?",
    all_chunks,
    k=3
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Question:
Who invented the telephone?

Answer:
The provided context does not contain any information about the invention of the telephone.

Sources:
['chocolate_history#0', 'chocolate_history#1', 'tea_black#0']


Next experiment — inspect the telephone similarity scores

In [25]:
QUERY = "Who invented the telephone?"

query_vector = embed_batch([QUERY])[0]

scored = []

for chunk in all_chunks:
    score = cosine(query_vector, chunk["vector"])
    scored.append((score, chunk))

scored.sort(key=lambda pair: pair[0], reverse=True)

print(f"Query: {QUERY}\n")

for rank, (score, chunk) in enumerate(scored[:10], start=1):
    print(
        f"{rank:>2}. "
        f"score={score:.3f}  "
        f"{chunk['chunk_id']}"
    )

Query: Who invented the telephone?

 1. score=0.185  chocolate_history#0
 2. score=0.158  chocolate_history#1
 3. score=0.085  tea_black#0
 4. score=0.058  coffee_brewing#0
 5. score=0.057  chocolate_powder#0
 6. score=0.052  tea_oolong#0
 7. score=0.051  coffee_brewing#1
 8. score=0.039  tea_green#0
 9. score=0.037  chocolate_powder#1
10. score=0.026  tea_black#1


Let's implement a threshold carefully

In [26]:
def retrieve_with_threshold(query, chunks, k=3, threshold=0.50):
    query_vector = embed_batch([query])[0]

    scored = []

    for chunk in chunks:
        score = cosine(query_vector, chunk["vector"])

        if score >= threshold:
            scored.append((score, chunk))

    scored.sort(key=lambda pair: pair[0], reverse=True)

    return [
        {**chunk, "score": score}
        for score, chunk in scored[:k]
    ]

Test the telephone question

In [27]:
telephone_results = retrieve_with_threshold(
    "Who invented the telephone?",
    all_chunks,
    k=3,
    threshold=0.50
)

print("Number of retrieved chunks:", len(telephone_results))

for rank, hit in enumerate(telephone_results, start=1):
    print(
        f"{rank}. {hit['chunk_id']} "
        f"score={hit['score']:.3f}"
    )

Number of retrieved chunks: 0


Now test the espresso question

In [28]:
espresso_results = retrieve_with_threshold(
    "how is espresso made?",
    all_chunks,
    k=3,
    threshold=0.50
)

print("Number of retrieved chunks:", len(espresso_results))

for rank, hit in enumerate(espresso_results, start=1):
    print(
        f"{rank}. {hit['chunk_id']} "
        f"score={hit['score']:.3f}"
    )

Number of retrieved chunks: 3
1. coffee_espresso#0 score=0.669
2. coffee_espresso#1 score=0.575
3. milk_latte#1 score=0.570


To modify  existing ask_rag() so it can handle the situation where retrieval returns zero chunks.

In [29]:
def ask_rag(question, chunks, k=3, threshold=None):
    # Step 1: retrieve relevant chunks
    if threshold is None:
        retrieved = retrieve(question, chunks, k=k)
    else:
        retrieved = retrieve_with_threshold(
            question,
            chunks,
            k=k,
            threshold=threshold
        )

    # Step 2: handle zero retrieved chunks
    if not retrieved:
        return {
            "question": question,
            "answer": (
                "I don't have enough information in the provided "
                "context to answer this question."
            ),
            "sources": []
        }

    # Step 3: build the grounded prompt
    system_message, user_message = build_prompt(
        question,
        retrieved
    )

    # Step 4: ask the LLM
    response = client.chat.completions.create(
        model=CHAT_MODEL,
        temperature=0.0,
        messages=[
            {
                "role": "system",
                "content": system_message
            },
            {
                "role": "user",
                "content": user_message
            }
        ]
    )

    # Step 5: extract the answer
    answer = response.choices[0].message.content

    # Step 6: return answer and retrieved sources
    return {
        "question": question,
        "answer": answer,
        "sources": [hit["chunk_id"] for hit in retrieved]
    }

Now test the telephone question

In [30]:
result = ask_rag(
    "Who invented the telephone?",
    all_chunks,
    k=3,
    threshold=0.50
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Question:
Who invented the telephone?

Answer:
I don't have enough information in the provided context to answer this question.

Sources:
[]


Then test espresso

In [31]:
result = ask_rag(
    "how is espresso made?",
    all_chunks,
    k=3,
    threshold=0.50
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Question:
how is espresso made?

Answer:
Espresso is made by forcing hot water under about 9 bars of pressure through finely ground coffee beans. A single shot is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract [coffee_espresso#0].

Sources:
['coffee_espresso#0', 'coffee_espresso#1', 'milk_latte#1']


                    QUESTION
                       │
                       ▼
                 Query embedding
                       │
                       ▼
             Compare 20 chunks
                       │
                       ▼
              Apply threshold
                       │
                ┌──────┴──────┐
                │             │
             chunks        no chunks
                │             │
                ▼             ▼
              Top-K       Safe response
                │
                ▼
             Prompt
                │
                ▼
               LLM
                │
                ▼
             Answer

Failure Mode: Garbled Query

In [32]:
result = ask_rag(
    "expresso how mak??",
    all_chunks,
    k=3
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Question:
expresso how mak??

Answer:
The context does not provide specific instructions on how to make espresso. It only describes what espresso is and its characteristics.

Sources:
['coffee_espresso#0', 'coffee_espresso#1', 'milk_latte#1']


Day 1, Failure Mode 9b

In [33]:
sys_msg, user_msg = build_prompt(
    "how is espresso made?",
    []
)

print("========== USER MESSAGE WITH ZERO CONTEXT ==========")
print(user_msg)

print("\n========== ASKING THE LLM ==========")

resp = client.chat.completions.create(
    model=CHAT_MODEL,
    temperature=0.0,
    messages=[
        {
            "role": "system",
            "content": sys_msg
        },
        {
            "role": "user",
            "content": user_msg
        }
    ]
)

print("\n========== ANSWER ==========")
print(resp.choices[0].message.content)

========== USER MESSAGE WITH ZERO CONTEXT ==========
Context:


---

Question: how is espresso made?

========== ASKING THE LLM ==========



========== ANSWER ==========
The provided context does not contain information on how espresso is made.


9d — Partial/mixed relevance

In [34]:
result = ask_rag(
    "How is espresso made, and how does it differ from a latte?",
    all_chunks,
    k=3
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
print(result["sources"])

Question:
How is espresso made, and how does it differ from a latte?

Answer:
Espresso is made by forcing hot water under about 9 bars of pressure through finely ground coffee beans, resulting in a concentrated form of coffee. A single shot of espresso is typically 25 to 30 millilitres and takes 25 to 30 seconds to extract [coffee_espresso#0].

A latte, on the other hand, is made with one shot of espresso and around 200 millilitres of steamed milk, topped with a thin layer of microfoam. The ratio of espresso to milk in a latte is roughly one part espresso to five parts milk [milk_latte#0]. Thus, the main difference is that espresso is a concentrated coffee drink, while a latte is a milk-based drink that includes espresso.

Sources:
['milk_latte#0', 'coffee_espresso#0', 'coffee_espresso#1']


Day 1 — Official Failure Mode 9d

In [35]:
result = ask_rag(
    "Compare the caffeine content of black tea and green tea, and the water "
    "temperatures used to brew them.",
    all_chunks,
    k=3
)

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources retrieved:")
print(result["sources"])

print("\nRetrieved chunks with content:\n")

for hit in retrieve(result["question"], all_chunks, k=3):
    print(f"[{hit['chunk_id']}] cosine={hit['score']:.3f}")
    print(f"  {hit['text'][:120]}...")
    print()

Question:
Compare the caffeine content of black tea and green tea, and the water temperatures used to brew them.

Answer:
Black tea typically contains more caffeine than green tea [tea_black#1]. Green tea is steeped in water at around 70 to 80 degrees Celsius [tea_green#0]. The context does not provide the specific water temperature for brewing black tea.

Sources retrieved:
['tea_black#1', 'tea_green#0', 'tea_green#1']

Retrieved chunks with content:

[tea_black#1] cosine=0.618
  . Popular varieties include Assam, Darjeeling, and Ceylon. Black tea typically contains more caffeine than green tea....

[tea_green#0] cosine=0.534
  Green tea is made from unoxidised leaves of Camellia sinensis. It is steeped in water at around 70 to 80 degrees Celsius...

[tea_green#1] cosine=0.525
  or longer steeping produces a bitter, astringent cup. Green tea is high in an antioxidant called EGCG....



Documents
    ↓
Chunking
    ↓
Embeddings
    ↓
Cosine similarity
    ↓
Top-K retrieval
    ↓
Prompt construction
    ↓
LLM generation
    ↓
Grounded answer

           RETRIEVAL
              │
              ▼
       Relevant chunks
              │
              │ but incomplete
              ▼
          GENERATION
              │
              ▼
    Answer what is supported
    + acknowledge what's missing